## Классификация редких событий (Imbalanced Rare-Hit Classification)

Условия дизбаланса

1. **Стратифицированное разбиение (`Stratified Split`)**: сохранение точной пропорции редкого класса в обучающей и тестовой выборках.
2. **Обогащение признаками аномалий**: используем непрерывный скор `score_iforest`, найденный на этапе поиска аномалий без учителя.
3. **Градиентный бустинг со взвешиванием классов (`class_weight='balanced'`)**.
4. **Метрики качества для редких событий**: Precision-Recall AUC (PR-AUC), ROC-AUC и матрица стоимости ошибок (Cost Matrix).

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Загрузка обогащенного датасета из Фазы 2
df = pd.read_csv('data/movies_with_anomalies.csv')
print(f"Датасет успешно загружен: {len(df):,} строк, {df.shape[1]} колонок")

# 2. Формирование признаков X и целевой переменной y
# Включаем базовые признаки фильма + скор аномальности из Isolation Forest
feature_cols = [
    'runtimeMinutes',
    'movie_age',
    'log_votes',
    'rating_to_log_votes',
    'score_iforest'
]

X = df[feature_cols].copy()
y = df['is_masterpiece'].copy()

# 3. Анализ экстремального дисбаланса классов
pos_count = y.sum()
neg_count = len(y) - pos_count
print(f"\nАнализ дисбаланса классов (целевая метка is_masterpiece):")
print(f"  Класс 0 (Обычные фильмы): {neg_count:,} ({neg_count / len(y):.2%})")
print(f"  Класс 1 (Редкие шедевры): {pos_count:,} ({pos_count / len(y):.3%})")
print(f"  Соотношение классов:      1 : {neg_count // pos_count}")

# 4. Стратифицированное разбиение на Train (80%) и Test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    train_size=0.8,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"\nРазмеры выборок после стратификации:")
print(f"  Train: {X_train.shape[0]:,} объектов (из них редких событий: {y_train.sum():,}, {y_train.mean():.3%})")
print(f"  Test:  {X_test.shape[0]:,} объектов (из них редких событий: {y_test.sum():,}, {y_test.mean():.3%})")

Датасет успешно загружен: 313,025 строк, 15 колонок

Анализ дисбаланса классов (целевая метка is_masterpiece):
  Класс 0 (Обычные фильмы): 312,470 (99.82%)
  Класс 1 (Редкие шедевры): 555 (0.177%)
  Соотношение классов:      1 : 563

Размеры выборок после стратификации:
  Train: 250,420 объектов (из них редких событий: 444, 0.177%)
  Test:  62,605 объектов (из них редких событий: 111, 0.177%)
